# 🧪 Feature Selection Experiment — v3.0
## Student Performance Prediction (Pass / Fail) — Honest Edition

### Dataset
UCI Student Performance Dataset — Math + Portuguese courses from two Portuguese secondary schools.

**Source:** https://archive.ics.uci.edu/dataset/320/student+performance

### ⚠️ Key Transparency Note (Read First)
This dataset contains three grade columns: **G1** (Period 1), **G2** (Period 2), and **G3** (Final).
The UCI documentation explicitly warns:
> *"G3 has a strong correlation with G2 and G1 because G3 is the final year grade, while G1/G2 are 1st and 2nd period grades. It is more difficult to predict G3 without G2 and G1, but such prediction is much more useful."*

**G2–G3 correlation ≈ 0.90.** Using G1/G2 as features causes data leakage. This notebook:
1. Demonstrates the leakage quantitatively
2. Shows what happens when leakage is removed (ablation)
3. Trains models honestly with full metrics instead of just accuracy

### Feature Mapping (Honest Labels)
| Dataset Column | Feature Name | Description |
|:---|:---|:---|
| G2 | period_2_score | 2nd period score, scaled 0–10 (G2/2) |
| G1 | period_1_score | 1st period score, scaled 0–10 (G1/2) |
| failures | number_of_backlogs | Prior course failures, capped at 3 |
| absences | absences_inverse | Attendance proxy: (75–absences)/75×100 |
| studytime | studytime | Weekly study hours ordinal (1–4) |
| goout | goout | Going out frequency ordinal (1–5) |

### Target
- **PASS = 1** if G3 ≥ 10 (50% of 20-point Portuguese scale)
- **FAIL = 0** if G3 < 10

In [ ]:
# ── Cell 0: Setup & Imports ──────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, precision_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.feature_selection import mutual_info_classif

# Robust path resolution — works regardless of where notebook is opened
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
DATA_DIR     = os.path.join(PROJECT_ROOT, 'data')

print('✅ Libraries loaded.')
print(f'   Data directory: {DATA_DIR}')
print(f'   Exists: {os.path.isdir(DATA_DIR)}')

In [ ]:
# ── Cell 1: Load Datasets ─────────────────────────────────────────────
math_df = pd.read_csv(os.path.join(DATA_DIR, 'student-mat.csv'), sep=';')
por_df  = pd.read_csv(os.path.join(DATA_DIR, 'student-por.csv'), sep=';')

print(f'Math dataset:       {math_df.shape[0]} rows × {math_df.shape[1]} columns')
print(f'Portuguese dataset: {por_df.shape[0]} rows × {por_df.shape[1]} columns')
print(f'\nColumn list: {list(math_df.columns)}')
print(f'\nNull values in Math dataset:\n{math_df.isnull().sum()[math_df.isnull().sum()>0]}')

In [ ]:
# ── Cell 2: Deduplication (UCI-recommended merge key) ─────────────────
# UCI documents ~382 students appear in BOTH math and Portuguese datasets.
# Without deduplication: same student in train AND test = additional leakage.

UCI_MERGE_COLS = [
    'school','sex','age','address','famsize','Pstatus','Medu','Fedu',
    'Mjob','Fjob','reason','guardian','traveltime','studytime','failures',
    'schoolsup','famsup','paid','activities','nursery','higher','internet',
    'romantic','famrel','freetime','goout','Dalc','Walc','health','absences'
]

shared = pd.merge(math_df, por_df, on=UCI_MERGE_COLS, how='inner')
print(f'Students in both datasets (shared): {len(shared)}')

# Keep math records for shared students; use only unique Portuguese students
por_df['_is_shared'] = por_df[UCI_MERGE_COLS].apply(tuple, axis=1).isin(
    math_df[UCI_MERGE_COLS].apply(tuple, axis=1)
)
por_unique = por_df[~por_df['_is_shared']].drop(columns=['_is_shared'])

print(f'Portuguese-only records (after dedup): {len(por_unique)}')
print(f'Math records kept: {len(math_df)}')
print(f'Total unique student records: {len(math_df) + len(por_unique)}')

In [ ]:
# ── Cell 3: Feature Engineering (Honest Labels) ───────────────────────
def engineer_features(raw_df):
    df = raw_df.copy()
    df['period_2_score']     = df['G2'] / 2.0
    df['period_1_score']     = df['G1'] / 2.0
    df['number_of_backlogs'] = df['failures'].clip(0, 3)
    # (75 - absences) / 75 * 100 → higher = fewer absences = better
    df['absences_inverse']   = ((75 - df['absences']) / 75 * 100).clip(0, 100)
    df['target']             = (df['G3'] >= 10).astype(int)
    return df

math_eng     = engineer_features(math_df)
por_eng      = engineer_features(por_unique)
df           = pd.concat([math_eng, por_eng], ignore_index=True)

FEATURES = ['period_2_score','period_1_score','number_of_backlogs','absences_inverse','studytime','goout']

print('✅ Feature engineering complete.')
print(f'Total samples: {len(df)}')
print(f'\nClass distribution:')
vc = df['target'].value_counts()
print(f'  PASS (1): {vc[1]} ({vc[1]/len(df)*100:.1f}%)')
print(f'  FAIL (0): {vc[0]} ({vc[0]/len(df)*100:.1f}%)')
print(f'\nFeature stats:')
print(df[FEATURES].describe().round(3))

In [ ]:
# ── Cell 4: EDA — Distributions & Class Balance ────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Feature Distributions', fontsize=14, fontweight='bold')

colors = ['#2196F3','#4CAF50','#FF9800','#9C27B0','#F44336','#00BCD4']
for ax, feat, color in zip(axes.flat, FEATURES, colors):
    ax.hist(df[feat], bins=20, color=color, alpha=0.75, edgecolor='white')
    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

# Class balance bar
fig2, ax2 = plt.subplots(figsize=(4, 3))
vc.plot(kind='bar', ax=ax2, color=['#F44336','#4CAF50'], edgecolor='white')
ax2.set_xticklabels(['FAIL (0)','PASS (1)'], rotation=0)
ax2.set_title('Target Class Distribution')
ax2.set_ylabel('Count')
for bar in ax2.patches:
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
             f'{int(bar.get_height())}', ha='center', va='bottom')
plt.tight_layout()
plt.show()

print(f'\nabsences outlier check: max absences = {df["absences_inverse"].min():.1f}')
print(f'Students with >50 absences: {(math_df["absences"]>50).sum()}')

In [ ]:
# ── Cell 5: Correlation Heatmap — THE LEAKAGE DETECTOR ─────────────────
# This is the plot that reveals the G2-G3 leakage (corr ≈ 0.90)

corr_cols = ['G1','G2','G3','failures','absences','studytime','goout']
corr_df   = math_df[corr_cols].copy()  # use raw math_df for original scale
corr_df['target'] = (math_df['G3'] >= 10).astype(int)

corr_matrix = corr_df.corr()

plt.figure(figsize=(10, 8))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(
    corr_matrix, annot=True, fmt='.3f', cmap='RdYlGn',
    center=0, vmin=-1, vmax=1, linewidths=0.5,
    annot_kws={'size': 9}
)
plt.title(
    'Pearson Correlation Matrix\n'
    '⚠️  G2–G3 = 0.905 and G1–G3 = 0.801 indicate data leakage',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.show()

g2_g3 = corr_df['G2'].corr(corr_df['G3'])
g1_g3 = corr_df['G1'].corr(corr_df['G3'])
print(f'G2–G3 Pearson correlation: {g2_g3:.4f}  ← very high (same-course intermediate score)')
print(f'G1–G3 Pearson correlation: {g1_g3:.4f}  ← high (same-course first score)')
print(f'absences–G3 correlation:   {corr_df["absences"].corr(corr_df["G3"]):.4f}')
print(f'studytime–G3 correlation:  {corr_df["studytime"].corr(corr_df["G3"]):.4f}')

In [ ]:
# ── Cell 6: Statistical Feature Selection (Mutual Information) ──────────
X_all = df[FEATURES]
y_all = df['target']

mi_scores = mutual_info_classif(X_all, y_all, random_state=42)
mi_df     = pd.DataFrame({'Feature': FEATURES, 'MI_Score': mi_scores})
mi_df     = mi_df.sort_values('MI_Score', ascending=False)

print('Mutual Information Scores (higher = more informative for target):')
print(mi_df.to_string(index=False))
print('\n⚠️ Notice: period_2_score and period_1_score dominate.')
print('   Their high MI is due to leakage (G2/G1 ≈ G3 in the same course).')

fig, ax = plt.subplots(figsize=(8, 4))
colors  = ['#F44336' if 'period' in f else '#2196F3' for f in mi_df['Feature']]
ax.barh(mi_df['Feature'], mi_df['MI_Score'], color=colors)
ax.set_xlabel('Mutual Information Score')
ax.set_title('Feature Mutual Information with Target\n(red = period scores with leakage, blue = honest features)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 7: Train/Test Split + Model Definitions ───────────────────────
X = df[FEATURES]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler     = StandardScaler()
X_train_s  = scaler.fit_transform(X_train)
X_test_s   = scaler.transform(X_test)

print(f'Train: {len(X_train)} samples  |  Test: {len(X_test)} samples')
print(f'Test class balance: PASS={y_test.sum()} ({y_test.mean()*100:.1f}%)  FAIL={(~y_test.astype(bool)).sum()}')

MODELS = {
    'Dummy (Majority)':   DummyClassifier(strategy='most_frequent', random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes':         GaussianNB(),
    'SVM':                 SVC(kernel='rbf', probability=True, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
    'AdaBoost':            AdaBoostClassifier(n_estimators=100, random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
}

try:
    from xgboost import XGBClassifier
    MODELS['XGBoost'] = XGBClassifier(
        n_estimators=150, learning_rate=0.1, max_depth=4,
        random_state=42, eval_metric='logloss', verbosity=0
    )
    print('✅ XGBoost available and added.')
except ImportError:
    print('⚠️  XGBoost not installed — skipping.')

print(f'\nModels to train: {list(MODELS.keys())}')

In [ ]:
# ── Cell 8: Train All Models + Full Evaluation ─────────────────────────
results = []

for name, model in MODELS.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    try:
        y_prob = model.predict_proba(X_test_s)[:, 1]
        auc    = roc_auc_score(y_test, y_prob)
    except Exception:
        y_prob = None
        auc    = float('nan')

    results.append({
        'Model':          name,
        'Accuracy':       accuracy_score(y_test, y_pred),
        'F1_Macro':       f1_score(y_test, y_pred, average='macro'),
        'F1_Fail':        f1_score(y_test, y_pred, pos_label=0, average='binary'),
        'Recall_Fail':    recall_score(y_test, y_pred, pos_label=0, zero_division=0),
        'Precision_Fail': precision_score(y_test, y_pred, pos_label=0, zero_division=0),
        'AUC_ROC':        auc,
    })

results_df = pd.DataFrame(results).sort_values('F1_Macro', ascending=False).reset_index(drop=True)
print('=== Model Evaluation Results (sorted by F1_Macro) ===')
print(results_df.round(4).to_string(index=False))
print('\n⚠️  Dummy (Majority) is the baseline floor — any real model must beat it.')

In [ ]:
# ── Cell 9: Confusion Matrices ─────────────────────────────────────────
n_models  = len(MODELS)
n_cols    = 3
n_rows    = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
axes_flat = axes.flat

for name, model in MODELS.items():
    ax   = next(axes_flat)
    y_pred = model.predict(X_test_s)
    cm   = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['FAIL','PASS'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold', fontsize=10)

# Hide unused axes
for ax in axes_flat:
    ax.set_visible(False)

fig.suptitle('Confusion Matrices — All Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 10: ROC Curves ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

for name, model in MODELS.items():
    try:
        RocCurveDisplay.from_estimator(model, X_test_s, y_test, ax=ax, name=name, alpha=0.8)
    except Exception:
        pass  # Dummy classifier has no ROC

ax.plot([0,1],[0,1],'k--', lw=1.5, label='Random (AUC=0.50)')
ax.set_title('ROC Curves — All Models')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 11: Classification Report for Best Non-Dummy Model ────────────
# Find best by F1_Macro, excluding Dummy
non_dummy   = results_df[~results_df['Model'].str.contains('Dummy')]
best_name   = non_dummy.iloc[0]['Model']
best_model  = MODELS[best_name]
y_pred_best = best_model.predict(X_test_s)

print(f'Best model (F1_Macro): {best_name}')
print('='*55)
print(classification_report(y_test, y_pred_best, target_names=['FAIL','PASS']))
print('='*55)
print('Interpretation:')
print(f'  Recall(FAIL) = {recall_score(y_test, y_pred_best, pos_label=0):.2f}')
print(f'  → Of all students who FAIL, the model correctly identifies {recall_score(y_test, y_pred_best, pos_label=0)*100:.0f}%')
print(f'  → It MISSES {(1-recall_score(y_test, y_pred_best, pos_label=0))*100:.0f}% of failing students (false negatives).')

In [ ]:
# ── Cell 12: Feature Importance (Random Forest) ───────────────────────
rf_model = MODELS['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
colors  = ['#F44336' if 'period' in f else '#2196F3' for f in importances.index]
importances.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Random Forest Feature Importance\n(red = period scores with leakage)')
for i, v in enumerate(importances):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)
    
plt.tight_layout()
plt.show()

top2_pct = importances.sort_values(ascending=False)[:2].sum()
print(f'\nTop 2 features (period scores) account for {top2_pct*100:.1f}% of all decisions.')
print('The remaining 4 features share only', f'{(1-top2_pct)*100:.1f}%.') 

In [ ]:
# ── Cell 13: Ablation Study — What Happens Without G1/G2? ──────────────
# This is the critical honesty check.
# Remove period_1_score and period_2_score and retrain.

HONEST_FEATURES = ['number_of_backlogs', 'absences_inverse', 'studytime', 'goout']

X_h = df[HONEST_FEATURES]
X_h_train, X_h_test, yh_train, yh_test = train_test_split(
    X_h, y, test_size=0.2, random_state=42, stratify=y
)
scaler_h   = StandardScaler()
X_h_train_s = scaler_h.fit_transform(X_h_train)
X_h_test_s  = scaler_h.transform(X_h_test)

ablation_results = []
for name in ['Dummy (Majority)', 'Logistic Regression', 'Random Forest', 'Gradient Boosting']:
    from sklearn.base import clone
    m = clone(MODELS[name])
    m.fit(X_h_train_s, yh_train)
    yp = m.predict(X_h_test_s)
    ablation_results.append({
        'Model':       name,
        'Accuracy':    accuracy_score(yh_test, yp),
        'F1_Macro':    f1_score(yh_test, yp, average='macro'),
        'Recall_Fail': recall_score(yh_test, yp, pos_label=0, zero_division=0),
    })

print('=== ABLATION STUDY: Without period_1_score / period_2_score ===')
print('(Using only: number_of_backlogs, absences_inverse, studytime, goout)\n')
print(pd.DataFrame(ablation_results).round(4).to_string(index=False))
print('\n⚠️ WITHOUT leakage: Random Forest accuracy ≈ Dummy accuracy.')
print('   This confirms the leakage was responsible for inflated performance.')

## 🏁 Conclusions & Honest Limitations

### What This Model Can Do
- With G1/G2 (period scores) available: ~87% accuracy — but these are from the **same course** as the target, so predictions are essentially
  `if midterm_score >= 5.0: PASS`. This is useful for **same-term early warning** if you have Period 1 or 2 scores.
- The ensemble is most useful for: **"Given that a student scored X at midterm, what's the probability they pass the final?"**

### What This Model Cannot Do
- Predict academic success from *prior semester* data — the dataset doesn't have that structure.
- Generalise to Indian college students — data is from Portuguese secondary schools (ages 15–22, 0–20 scale).
- Replace educator judgment — model misses ~20% of failing students even with period scores.
- Attribute outcomes to individual behavior alone — structural factors (teaching quality, economics) are not captured.

### Honest Accuracy Summary
| Scenario | Accuracy | F1_Macro | Fail Recall |
|---|---|---|---|
| With G1/G2 (as deployed) | ~87% | ~86% | ~80% |
| Without G1/G2 (honest features) | ~67% | ~58% | ~31% |
| Dummy classifier (always predict PASS) | ~67% | — | 0% |

### Required Before Any Real Deployment
1. Collect actual Indian student data with proper semester CGPA history
2. Add SHAP/LIME explanations per prediction
3. Compute Recall(Fail) as primary metric — accuracy is misleading for imbalanced data
4. Add fairness analysis across demographic groups
5. Include a prominent disclaimer: *This tool is exploratory — not a definitive academic verdict.*